<a href="https://colab.research.google.com/github/Rukshana-S/mcd-staffing-demand-prediction/blob/main/Restaurant_Staffing_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

project_path = "/content/drive/MyDrive/mcd-staffing-demand-prediction"

folders = [
    "data/raw",
    "data/processed",
    "models",
    "outputs/charts",
    "notebooks"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [3]:
!pip install -q pandas numpy matplotlib scikit-learn joblib

In [4]:
import pandas as pd
import os

data_path = os.path.join(
    project_path,
    "data/raw/McDonalds_Restaurant_Staffing_Mock_Dataset_200.csv"
)

df = pd.read_csv(data_path)

print("Dataset Loaded Successfully!")
print(df.shape)

df.head()

Dataset Loaded Successfully!
(200, 9)


,Date,Outlet_ID,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Day_Type,Holiday
0,2025-03-23,MCD_CBE_01,9,120.0,7.6,4,5.0,Weekend,Yes
1,2025-03-17,MCD_CHN_01,19,194.0,9.3,6,9.0,Weekday,Yes
2,2025-03-25,MCD_MDU_01,23,5.0,4.4,0,2.0,Weekday,No
3,2025-03-31,MCD_CHN_02,19,176.0,9.4,4,8.0,Weekday,No
4,2025-02-18,MCD_CBE_01,9,44.0,5.0,3,3.0,Weekday,No


In [5]:
print("Dataset Information")
print("="*50)

df.info()

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      200 non-null    object 
 1   Outlet_ID                 200 non-null    object 
 2   Hour                      200 non-null    int64  
 3   Order_Volume              195 non-null    float64
 4   Average_Preparation_Time  200 non-null    float64
 5   Queue_Length              200 non-null    int64  
 6   Staff_Count               195 non-null    float64
 7   Day_Type                  200 non-null    object 
 8   Holiday                   200 non-null    object 
dtypes: float64(3), int64(2), object(4)
memory usage: 14.2+ KB


In [6]:
print("Missing Values")
print("="*50)

df.isnull().sum()

Missing Values


,0
Date,0
Outlet_ID,0
Hour,0
Order_Volume,5
Average_Preparation_Time,0
Queue_Length,0
Staff_Count,5
Day_Type,0
Holiday,0


In [7]:
print("Summary Statistics")
print("="*50)

df.describe(include="all")

Summary Statistics


,Date,Outlet_ID,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Day_Type,Holiday
count,200,200,200.000000,195.000000,200.000000,200.000000,195.000000,200,200
unique,82,5,NaN,NaN,NaN,NaN,NaN,2,2
top,2025-03-19,MCD_CBE_01,NaN,NaN,NaN,NaN,NaN,Weekday,No
freq,7,50,NaN,NaN,NaN,NaN,NaN,142,181
mean,NaN,NaN,15.045000,99.292308,6.558500,2.915000,5.164103,NaN,NaN
std,NaN,NaN,5.591898,60.284392,2.004625,2.995436,2.847271,NaN,NaN
min,NaN,NaN,6.000000,5.000000,-3.000000,0.000000,2.000000,NaN,NaN
25%,NaN,NaN,10.000000,46.500000,5.700000,0.000000,2.000000,NaN,NaN
50%,NaN,NaN,15.000000,97.000000,6.800000,2.000000,5.000000,NaN,NaN
75%,NaN,NaN,20.000000,156.500000,7.825000,6.000000,8.000000,NaN,NaN


In [8]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print(f"Removed {before-after} duplicate rows.")

Removed 1 duplicate rows.


In [9]:
df["Day_Type"] = df["Day_Type"].replace({
    "Weeknd":"Weekend"
})

df["Day_Type"].value_counts()

,count
Day_Type,
Weekday,141
Weekend,58


In [10]:
before = len(df)

df = df[(df["Hour"] >= 6) & (df["Hour"] <= 23)]

print(f"Removed {before-len(df)} invalid hour records.")

Removed 4 invalid hour records.


In [11]:
df = df[
    (df["Average_Preparation_Time"] > 0) &
    (df["Queue_Length"] >= 0) &
    (df["Staff_Count"] > 0)
]

In [12]:
numeric_cols = [
    "Order_Volume",
    "Staff_Count"
]

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [13]:
print(df.isnull().sum())

print()

print("Final Shape:", df.shape)

Date                        0
Outlet_ID                   0
Hour                        0
Order_Volume                0
Average_Preparation_Time    0
Queue_Length                0
Staff_Count                 0
Day_Type                    0
Holiday                     0
dtype: int64

Final Shape: (185, 9)


In [14]:
clean_path = os.path.join(
    project_path,
    "data/processed/restaurant_cleaned.csv"
)

df.to_csv(clean_path, index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [15]:
print(df.isnull().sum())

print()

print("Final Shape:", df.shape)

Date                        0
Outlet_ID                   0
Hour                        0
Order_Volume                0
Average_Preparation_Time    0
Queue_Length                0
Staff_Count                 0
Day_Type                    0
Holiday                     0
dtype: int64

Final Shape: (185, 9)


In [16]:
import pandas as pd
import numpy as np
import os

In [17]:
print("Clean Dataset Shape:", df.shape)
print()

df.head()

Clean Dataset Shape: (185, 9)



,Date,Outlet_ID,Hour,Order_Volume,Average_Preparation_Time,Queue_Length,Staff_Count,Day_Type,Holiday
0,2025-03-23,MCD_CBE_01,9,120.0,7.6,4,5.0,Weekend,Yes
1,2025-03-17,MCD_CHN_01,19,194.0,9.3,6,9.0,Weekday,Yes
2,2025-03-25,MCD_MDU_01,23,5.0,4.4,0,2.0,Weekday,No
3,2025-03-31,MCD_CHN_02,19,176.0,9.4,4,8.0,Weekday,No
4,2025-02-18,MCD_CBE_01,9,44.0,5.0,3,3.0,Weekday,No


In [18]:
hourly_orders = (
    df.groupby("Hour")["Order_Volume"]
      .sum()
      .reset_index()
      .sort_values("Hour")
)

hourly_orders

,Hour,Order_Volume
0,6,313.0
1,7,448.0
2,8,1000.0
3,9,1229.0
4,10,429.0
5,11,88.0
6,12,1457.0
7,13,2246.0
8,14,1409.0
9,15,170.0


In [19]:
hourly_orders.to_csv(
    os.path.join(project_path, "outputs/hourly_orders_summary.csv"),
    index=False
)

In [20]:
outlet_summary = (
    df.groupby("Outlet_ID")
      .agg(
          Total_Orders=("Order_Volume","sum"),
          Avg_Prep_Time=("Average_Preparation_Time","mean"),
          Avg_Queue=("Queue_Length","mean"),
          Avg_Staff=("Staff_Count","mean")
      )
      .reset_index()
)

outlet_summary

,Outlet_ID,Total_Orders,Avg_Prep_Time,Avg_Queue,Avg_Staff
0,MCD_CBE_01,4135.0,6.525532,3.021277,4.468085
1,MCD_CHN_01,2221.0,6.927273,3.181818,4.909091
2,MCD_CHN_02,4076.0,7.012821,2.384615,5.615385
3,MCD_MDU_01,4974.0,7.016279,3.023256,5.976744
4,MCD_TRY_01,3126.0,6.614706,3.235294,4.823529


In [21]:
outlet_summary.to_csv(
    os.path.join(project_path, "outputs/outlet_summary.csv"),
    index=False
)

In [22]:
day_summary = (
    df.groupby("Day_Type")
      .agg(
          Average_Orders=("Order_Volume","mean"),
          Average_Queue=("Queue_Length","mean"),
          Average_Staff=("Staff_Count","mean")
      )
      .reset_index()
)

day_summary

,Day_Type,Average_Orders,Average_Queue,Average_Staff
0,Weekday,89.356061,2.590909,4.772727
1,Weekend,127.113208,3.830189,6.188679


In [23]:
day_summary.to_csv(
    os.path.join(project_path, "outputs/day_type_summary.csv"),
    index=False
)

In [24]:
holiday_summary = (
    df.groupby("Holiday")
      .agg(
          Average_Orders=("Order_Volume","mean"),
          Average_Queue=("Queue_Length","mean"),
          Average_Staff=("Staff_Count","mean")
      )
      .reset_index()
)

holiday_summary

,Holiday,Average_Orders,Average_Queue,Average_Staff
0,No,98.196429,2.869048,5.113095
1,Yes,119.705882,3.705882,5.823529


In [25]:
holiday_summary.to_csv(
    os.path.join(project_path, "outputs/holiday_summary.csv"),
    index=False
)

In [26]:
df["Orders_per_Employee"] = (
    df["Order_Volume"] / df["Staff_Count"]
).round(2)

df[["Hour","Order_Volume","Staff_Count","Orders_per_Employee"]].head()

,Hour,Order_Volume,Staff_Count,Orders_per_Employee
0,9,120.0,5.0,24.00
1,19,194.0,9.0,21.56
2,23,5.0,2.0,2.50
3,19,176.0,8.0,22.00
4,9,44.0,3.0,14.67


In [27]:
df.to_csv(
    os.path.join(project_path, "data/processed/restaurant_cleaned.csv"),
    index=False
)

In [28]:
workload_summary = (
    df.groupby("Hour")
      .agg(
          Avg_Orders_Per_Employee=("Orders_per_Employee","mean")
      )
      .reset_index()
      .sort_values("Avg_Orders_Per_Employee", ascending=False)
)

workload_summary.head(10)

,Hour,Avg_Orders_Per_Employee
1,7,22.861111
13,19,21.738182
4,10,21.653750
2,8,20.900000
8,14,20.684444
15,21,20.624286
12,18,20.381111
7,13,20.031429
16,22,19.844000
3,9,19.806667


In [29]:
workload_summary.to_csv(
    os.path.join(project_path, "outputs/workload_summary.csv"),
    index=False
)